# Fine-tune YOLOv8 cho 6 class gesture (Tesla V100)
Notebook này chỉ set-up flow code, không tự chạy train.
Bạn có thể chạy từng cell trên máy thuê GPU Tesla V100-SXM2-32GB.

In [7]:
# Cài thư viện cần thiết
# Trên server mới, chạy cell này trước
!pip install -q ultralytics pandas matplotlib pyyaml

## 1) Kiểm tra môi trường GPU và phiên bản thư viện

In [8]:
# Cài thư viện cho môi trường server/headless (không có GUI)
# Mục tiêu: tránh lỗi ImportError: libGL.so.1 khi import ultralytics/cv2
!pip install -q --upgrade pip
!pip uninstall -y opencv-python opencv-contrib-python opencv-contrib-python-headless
!pip install -q --no-cache-dir --force-reinstall "opencv-python-headless>=4.10.0" "ultralytics>=8.3.0" pandas matplotlib pyyaml

# Nếu máy cho phép apt-get, cài thêm runtime cần cho OpenCV

print('Xong cài đặt. Nếu trước đó đã import lỗi, hãy Restart Kernel rồi chạy lại từ Cell 1.')

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Xong cài đặt. Nếu trước đó đã import lỗi, hãy Restart Kernel rồi chạy lại từ Cell 1.


In [37]:
# Root thư mục project
from pathlib import Path

PROJECT_ROOT = Path.cwd()
ZIP_PATH = PROJECT_ROOT / 'dataset.zip'
DATASET_ROOT = PROJECT_ROOT / 'dataset'
ORIGINAL_YAML = DATASET_ROOT / 'data.yaml'

# Nếu chưa có dataset/data.yaml thì giải nén từ dataset.zip
if not ORIGINAL_YAML.exists() and ZIP_PATH.exists():
    import zipfile

    print(f'Đang giải nén: {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(PROJECT_ROOT)

# Thử lại đường dẫn mặc định sau giải nén
DATASET_ROOT = PROJECT_ROOT / 'dataset'
ORIGINAL_YAML = DATASET_ROOT / 'data.yaml'

# Fallback: nếu zip giải nén ra thư mục khác, tự tìm data.yaml
if not ORIGINAL_YAML.exists():
    found = list(PROJECT_ROOT.rglob('data.yaml'))
    found = [p for p in found if 'dataset' in p.as_posix()]
    if found:
        ORIGINAL_YAML = found[0]
        DATASET_ROOT = ORIGINAL_YAML.parent

assert ORIGINAL_YAML.exists(), (
    f'Không tìm thấy data.yaml. Kiểm tra file zip tại: {ZIP_PATH}'
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('ZIP_PATH     =', ZIP_PATH)
print('DATASET_ROOT =', DATASET_ROOT)
print('ORIGINAL_YAML =', ORIGINAL_YAML)

PROJECT_ROOT = /workspace
ZIP_PATH     = /workspace/dataset.zip
DATASET_ROOT = /workspace/dataset
ORIGINAL_YAML = /workspace/dataset/data.yaml


In [52]:
# Tạo YAML runtime theo đúng cấu trúc thực tế trên server
import yaml
from pathlib import Path


def detect_split_dir(root: Path, split: str) -> str:
    candidates = [
        f'images/{split}',
        f'image/{split}',
        f'imgs/{split}',
        f'lables/{split}',   # typo thường gặp
        f'labels/{split}',   # fallback nếu dữ liệu ảnh nằm nhầm trong labels
    ]
    for rel in candidates:
        p = root / rel
        if p.exists() and p.is_dir():
            return rel
    raise FileNotFoundError(f'Không tìm thấy thư mục cho split={split}. Đã thử: {candidates}')


with open(ORIGINAL_YAML, 'r', encoding='utf-8') as f:
    train_cfg_yaml = yaml.safe_load(f)

train_rel = detect_split_dir(DATASET_ROOT, 'train')
val_rel = detect_split_dir(DATASET_ROOT, 'val')

train_cfg_yaml['path'] = str(DATASET_ROOT.resolve())
train_cfg_yaml['train'] = train_rel
train_cfg_yaml['val'] = val_rel

TRAIN_DATA_YAML = PROJECT_ROOT / 'dataset' / 'data_runtime.yaml'
with open(TRAIN_DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(train_cfg_yaml, f, sort_keys=False, allow_unicode=True)

print('Original yaml:', ORIGINAL_YAML)
print('Runtime  yaml:', TRAIN_DATA_YAML)
print('Runtime path :', train_cfg_yaml['path'])
print('Runtime train:', train_cfg_yaml['train'])
print('Runtime val  :', train_cfg_yaml['val'])

Original yaml: /workspace/dataset/data.yaml
Runtime  yaml: /workspace/dataset/data_runtime.yaml
Runtime path : /workspace/dataset
Runtime train: images/train
Runtime val  : images/val


In [53]:
# Cấu hình train cho Tesla V100 32GB
PRETRAINED_WEIGHTS = 'yolov8n.pt'  # có thể đổi sang yolov8s.pt nếu muốn mô hình lớn hơn
RUN_NAME = 'gesture_v100_ft'
PROJECT_DIR = PROJECT_ROOT / 'runs' / 'detect'

TRAIN_CFG = {
    'data': str(TRAIN_DATA_YAML),  # sẽ được ép lại ở Cell train để tránh dùng config cũ trong kernel
    'epochs': 100,
    'imgsz': 640,
    'batch': 64,
    'device': 0,
    'workers': 8,
    'optimizer': 'SGD',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 5e-4,
    'warmup_epochs': 3.0,
    'cos_lr': True,
    'patience': 20,
    'amp': True,
    'project': str(PROJECT_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
}

print('Train config:')
for k, v in TRAIN_CFG.items():
    print(f'- {k}: {v}')

Train config:
- data: /workspace/dataset/data_runtime.yaml
- epochs: 100
- imgsz: 640
- batch: 64
- device: 0
- workers: 8
- optimizer: SGD
- lr0: 0.01
- lrf: 0.01
- momentum: 0.937
- weight_decay: 0.0005
- warmup_epochs: 3.0
- cos_lr: True
- patience: 20
- amp: True
- project: /workspace/runs/detect
- name: gesture_v100_ft
- exist_ok: True


In [54]:
# Chạy train khi đã sẵn sàng trên server GPU
RUN_TRAIN = True

import yaml
from pathlib import Path


def detect_split_dir(root: Path, split: str) -> str:
    candidates = [
        f'images/{split}',
        f'image/{split}',
        f'imgs/{split}',
        f'lables/{split}',
        f'labels/{split}',
    ]
    for rel in candidates:
        p = root / rel
        if p.exists() and p.is_dir():
            return rel
    raise FileNotFoundError(f'Không tìm thấy thư mục cho split={split}. Đã thử: {candidates}')


# Luôn tạo lại YAML runtime ngay trước train để tránh dùng path cũ từ kernel trước đó
with open(ORIGINAL_YAML, 'r', encoding='utf-8') as f:
    train_cfg_yaml = yaml.safe_load(f)

train_rel = detect_split_dir(DATASET_ROOT, 'train')
val_rel = detect_split_dir(DATASET_ROOT, 'val')

train_cfg_yaml['path'] = str(DATASET_ROOT.resolve())
train_cfg_yaml['train'] = train_rel
train_cfg_yaml['val'] = val_rel

TRAIN_DATA_YAML = PROJECT_ROOT / 'dataset' / 'data_runtime.yaml'
with open(TRAIN_DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(train_cfg_yaml, f, sort_keys=False, allow_unicode=True)

# Validate path train/val trước khi gọi YOLO.train
train_dir = Path(train_cfg_yaml['path']) / train_cfg_yaml['train']
val_dir = Path(train_cfg_yaml['path']) / train_cfg_yaml['val']
assert train_dir.exists(), f'Train dir không tồn tại: {train_dir}'
assert val_dir.exists(), f'Val dir không tồn tại: {val_dir}'

print('Using data yaml:', TRAIN_DATA_YAML)
print('Train dir:', train_dir)
print('Val dir  :', val_dir)

# Guard: tự xử lý môi trường nếu import ultralytics lỗi libGL/cv2
try:
    from ultralytics import YOLO
except Exception as e:
    print(f'Import ultralytics lỗi: {e}')
    print('Đang thử cài thêm thư viện hệ thống + headless OpenCV...')

    import subprocess
    import sys

    subprocess.run(
        [
            'bash',
            '-lc',
            'if command -v apt-get >/dev/null 2>&1; then apt-get update -y && apt-get install -y libgl1 libglib2.0-0; else echo "apt-get not available, skip"; fi',
        ],
        check=False,
    )

    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python', 'opencv-contrib-python', 'opencv-contrib-python-headless'],
        check=False,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'opencv-python-headless>=4.10.0', 'ultralytics>=8.3.0'],
        check=True,
    )

    from ultralytics import YOLO

# Config train runtime: không phụ thuộc TRAIN_CFG cũ trong kernel
train_cfg_runtime = {
    'data': str(TRAIN_DATA_YAML),
    'epochs': 100,
    'imgsz': 640,
    'batch': 64,
    'device': 0,
    'workers': 8,
    'optimizer': 'SGD',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 5e-4,
    'warmup_epochs': 3.0,
    'cos_lr': True,
    'patience': 20,
    'amp': True,
    'project': str(PROJECT_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
}

model = YOLO(PRETRAINED_WEIGHTS)

if RUN_TRAIN:
    results = model.train(**train_cfg_runtime)
    print('Train xong. Kết quả nằm tại:', PROJECT_DIR / RUN_NAME)
else:
    print('Đang ở chế độ setup-only. Chưa chạy train.')
    print('Để train, đặt RUN_TRAIN=True rồi chạy lại cell này.')

Using data yaml: /workspace/dataset/data_runtime.yaml
Train dir: /workspace/dataset/images/train
Val dir  : /workspace/dataset/images/val
Ultralytics 8.4.24 🚀 Python-3.10.13 torch-2.10.0+cu128 CUDA:0 (Tesla V100-SXM2-32GB, 32494MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/dataset/data_runtime.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.

In [57]:
import matplotlib.pyplot as plt
# Cell theo dõi chỉ số học sau khi train (chạy cell này SAU khi đã train)
RESULTS_CSV = PROJECT_DIR / RUN_NAME / 'results.csv'

if not RESULTS_CSV.exists():
    print('Chưa có results.csv. Hãy train trước rồi chạy lại cell này.')
else:
    df = pd.read_csv(RESULTS_CSV)
    print('Số epoch đã log:', len(df))
    print('Các cột:', list(df.columns))

    def find_col(candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    col_epoch = find_col(['epoch'])
    col_map50 = find_col(['metrics/mAP50(B)', 'metrics/mAP50'])
    col_map95 = find_col(['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
    col_prec = find_col(['metrics/precision(B)', 'metrics/precision'])
    col_rec = find_col(['metrics/recall(B)', 'metrics/recall'])
    col_train_box = find_col(['train/box_loss'])
    col_train_cls = find_col(['train/cls_loss'])
    col_val_box = find_col(['val/box_loss'])
    col_val_cls = find_col(['val/cls_loss'])

    metric_cols = [c for c in [col_map50, col_map95, col_prec, col_rec] if c]
    if metric_cols:
        best_idx = df[col_map95].idxmax() if col_map95 else df[metric_cols[0]].idxmax()
        print('\nBest epoch summary:')
        print(df.loc[best_idx, [c for c in [col_epoch, col_map50, col_map95, col_prec, col_rec] if c]])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = df[col_epoch] if col_epoch else range(len(df))

    if col_map50: axes[0].plot(x, df[col_map50], label='mAP50')
    if col_map95: axes[0].plot(x, df[col_map95], label='mAP50-95')
    if col_prec: axes[0].plot(x, df[col_prec], label='Precision')
    if col_rec: axes[0].plot(x, df[col_rec], label='Recall')
    axes[0].set_title('Quality Metrics by Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].grid(True)
    axes[0].legend()

    if col_train_box: axes[1].plot(x, df[col_train_box], label='train/box_loss')
    if col_train_cls: axes[1].plot(x, df[col_train_cls], label='train/cls_loss')
    if col_val_box: axes[1].plot(x, df[col_val_box], label='val/box_loss')
    if col_val_cls: axes[1].plot(x, df[col_val_cls], label='val/cls_loss')
    axes[1].set_title('Loss Curves by Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    BEST_PT = PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'
    LAST_PT = PROJECT_DIR / RUN_NAME / 'weights' / 'last.pt'
    print('\nWeights path:')
    print('- best:', BEST_PT)
    print('- last:', LAST_PT)

Số epoch đã log: 100
Các cột: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

Best epoch summary:
epoch                   85.00000
metrics/mAP50(B)         0.99344
metrics/mAP50-95(B)      0.87623
metrics/precision(B)     0.99096
metrics/recall(B)        0.98995
Name: 84, dtype: float64


<Figure size 1400x500 with 2 Axes>


Weights path:
- best: /workspace/runs/detect/gesture_v100_ft/weights/best.pt
- last: /workspace/runs/detect/gesture_v100_ft/weights/last.pt


In [2]:
# 2) Setup inference model YOLOv8 đã train (chạy độc lập, không phụ thuộc cell trước)
from pathlib import Path
from ultralytics import YOLO

# Fallback an toàn nếu bạn chưa chạy các cell train config trong kernel hiện tại
PROJECT_ROOT = globals().get('PROJECT_ROOT', Path.cwd())
PROJECT_DIR = globals().get('PROJECT_DIR', PROJECT_ROOT / 'runs' / 'detect')
RUN_NAME = globals().get('RUN_NAME', 'gesture_v100_ft')

candidate_weights = [
    PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt',
    PROJECT_DIR / RUN_NAME / 'weights' / 'last.pt',
]

# Nếu chưa đúng RUN_NAME, tự tìm run gần nhất trong runs/detect
if not any(p.exists() for p in candidate_weights):
    best_candidates = sorted(
        PROJECT_DIR.glob('*/weights/best.pt'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    last_candidates = sorted(
        PROJECT_DIR.glob('*/weights/last.pt'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    candidate_weights += best_candidates + last_candidates

MODEL_PATH = next((p for p in candidate_weights if p.exists()), None)
assert MODEL_PATH is not None, (
    f'Không tìm thấy weight đã train trong: {PROJECT_DIR}. Hãy kiểm tra lại thư mục runs/detect/.../weights/'
)

model = YOLO(str(MODEL_PATH))
names = model.names
CLASS_NAMES = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)

print('Model dùng để test:', MODEL_PATH)
print('Số class:', len(CLASS_NAMES))
print('Class names:', CLASS_NAMES)

Model dùng để test: /Users/vananhduy/Documents/Repository_Git_Hub/Real-time-Spotify-Controller-via-Hand-Gesture-Recognition-using-YOLO/runs/detect/gesture_v100_ft/weights/best.pt
Số class: 7
Class names: ['fist', 'one', 'peace', 'three', 'four', 'palm', 'no_gesture']


In [4]:
# 3) Realtime test webcam trong notebook
import time
import cv2
import numpy as np
from IPython.display import display, clear_output, Image

if 'model' not in globals() or 'CLASS_NAMES' not in globals():
    raise RuntimeError('Bạn cần chạy Cell 10 trước để load model và class names.')

# Chế độ hiển thị:
# - 'inline': hiển thị ngay trong output của notebook (khuyên dùng trong VS Code/Jupyter)
# - 'window': mở cửa sổ cv2.imshow (hợp với script local, có GUI đầy đủ)
DISPLAY_MODE = 'inline'
CAMERA_INDEX = 0
CONF_THRES = 0.40
IMG_SIZE = 640
MAX_FPS = 15
MAX_SECONDS = 60  # tránh chạy vô hạn trong notebook; có thể tăng nếu muốn

# macOS thường ổn định hơn với AVFoundation
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_AVFOUNDATION)
if not cap.isOpened():
    cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError(
        f'Không mở được camera index={CAMERA_INDEX}. Thử đổi CAMERA_INDEX (0/1/2) hoặc kiểm tra quyền camera.'
    )

print(f'Bắt đầu realtime inference (mode={DISPLAY_MODE})...')
if DISPLAY_MODE == 'inline':
    print('Dừng bằng Interrupt Kernel (nút Stop) hoặc chờ hết MAX_SECONDS.')
else:
    print('Nhấn q hoặc ESC để thoát cửa sổ.')

start_t = time.time()
prev_t = start_t
frame_interval = 1.0 / max(MAX_FPS, 1)

try:
    while True:
        now = time.time()
        if now - start_t > MAX_SECONDS:
            print('Đã đạt MAX_SECONDS, tự dừng.')
            break

        ret, frame = cap.read()
        if not ret or frame is None:
            print('Không đọc được frame từ camera. Dừng.')
            break

        results = model.predict(source=frame, conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)
        r0 = results[0]
        annotated = r0.plot()

        detected_names = []
        if r0.boxes is not None and len(r0.boxes) > 0:
            cls_ids = r0.boxes.cls.detach().cpu().numpy().astype(int).tolist()
            detected_names = [CLASS_NAMES[c] if c < len(CLASS_NAMES) else str(c) for c in cls_ids]

        fps = 1.0 / max(now - prev_t, 1e-6)
        prev_t = now

        text_detect = 'Detected: ' + (', '.join(sorted(set(detected_names))) if detected_names else 'None')
        cv2.putText(annotated, f'FPS: {fps:.1f}', (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20, 220, 20), 2, cv2.LINE_AA)
        cv2.putText(annotated, text_detect, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2, cv2.LINE_AA)

        if DISPLAY_MODE == 'window':
            cv2.imshow('YOLOv8 Gesture Realtime Test', annotated)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                break
        else:
            ok, jpg = cv2.imencode('.jpg', annotated)
            if ok:
                clear_output(wait=True)
                display(Image(data=jpg.tobytes()))
                print(f'FPS: {fps:.1f} | {text_detect}')
            time.sleep(frame_interval)

finally:
    cap.release()
    if DISPLAY_MODE == 'window':
        cv2.destroyAllWindows()
    print('Đã đóng webcam test.')

Bắt đầu realtime inference (mode=inline)...
Dừng bằng Interrupt Kernel (nút Stop) hoặc chờ hết MAX_SECONDS.
Không đọc được frame từ camera. Dừng.
Đã đóng webcam test.


In [5]:
# 4) Fix OpenCV GUI cho macOS local (để mở được cửa sổ webcam native)
import platform
import subprocess
import sys
import cv2

is_macos = platform.system() == 'Darwin'
build_info = cv2.getBuildInformation()
is_headless_build = ('headless' in build_info.lower()) or ('GUI:                           NONE' in build_info)

print('Platform:', platform.system())
print('OpenCV version:', cv2.__version__)
print('Headless build:', is_headless_build)

if is_macos and is_headless_build:
    print('Đang chuyển từ opencv-python-headless -> opencv-python (GUI)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python-headless'],
        check=False,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', 'opencv-python>=4.10.0'],
        check=True,
    )
    print('Đã cài xong opencv-python.')
    print('Quan trọng: hãy Restart Kernel, rồi chạy lại Cell 10 và Cell 13.')
else:
    print('OpenCV hiện tại đã sẵn sàng cho chế độ cửa sổ (hoặc không phải macOS).')

Platform: Darwin
OpenCV version: 4.12.0
Headless build: False
OpenCV hiện tại đã sẵn sàng cho chế độ cửa sổ (hoặc không phải macOS).


In [ ]:
# 5) Realtime detect bằng cửa sổ webcam native trên macOS (cv2.imshow)
import time
import platform
import cv2

if 'model' not in globals() or 'CLASS_NAMES' not in globals():
    raise RuntimeError('Bạn cần chạy Cell 10 trước để load model và class names.')

if platform.system() != 'Darwin':
    print('Cell này tối ưu cho macOS. Hệ điều hành hiện tại:', platform.system())

# Kiểm tra khả năng GUI của OpenCV
build_info = cv2.getBuildInformation()
if ('headless' in build_info.lower()) or ('GUI:                           NONE' in build_info):
    raise RuntimeError(
        'OpenCV hiện là headless nên không mở được cửa sổ. Hãy chạy Cell 12, Restart Kernel, rồi chạy lại Cell 10 -> Cell 13.'
    )

CAMERA_INDEX = 0
CONF_THRES = 0.40
IMG_SIZE = 640

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_AVFOUNDATION)
if not cap.isOpened():
    cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError(f'Không mở được camera index={CAMERA_INDEX}. Thử 0/1/2.')

cv2.namedWindow('YOLOv8 Gesture Realtime Test (macOS)', cv2.WINDOW_NORMAL)
cv2.resizeWindow('YOLOv8 Gesture Realtime Test (macOS)', 1100, 700)
print('Đã mở webcam native. Nhấn q hoặc ESC để thoát.')

prev_t = time.time()

try:
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print('Không đọc được frame từ camera. Dừng.')
            break

        results = model.predict(source=frame, conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)
        r0 = results[0]
        annotated = r0.plot()

        detected_names = []
        if r0.boxes is not None and len(r0.boxes) > 0:
            cls_ids = r0.boxes.cls.detach().cpu().numpy().astype(int).tolist()
            detected_names = [CLASS_NAMES[c] if c < len(CLASS_NAMES) else str(c) for c in cls_ids]

        now = time.time()
        fps = 1.0 / max(now - prev_t, 1e-6)
        prev_t = now

        text_detect = 'Detected: ' + (', '.join(sorted(set(detected_names))) if detected_names else 'None')
        cv2.putText(annotated, f'FPS: {fps:.1f}', (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20, 220, 20), 2, cv2.LINE_AA)
        cv2.putText(annotated, text_detect, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow('YOLOv8 Gesture Realtime Test (macOS)', annotated)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord('q'), 27):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print('Đã đóng webcam test.')